# YBS 4. Sınıf — Python ile Veri Bilimi: Dönem Sonu Projesi
## Konu: Maliyet Odaklı Müşteri Kaybı (Churn) Tahmini ve Finansal Simülasyon

**Öğrenci:** Fatmanur Sena Bülbül

**Tema:** Tema 2 — Hata Maliyeti Odaklı Müşteri Analitiği

---

### Proje Özeti
Bu projede bir telekomünikasyon şirketinin müşteri kaybını (churn) tahmin eden, standart %50 karar eşiği yerine iş maliyetlerine göre optimize edilmiş bir eşik kullanan ve finansal simülasyonla ROI hesaplayan bir makine öğrenmesi sistemi geliştirilmiştir.

**Zorunlu Kriterler:**
- ✅ **Veri Harmanlama:** IBM Telco müşteri verisi + TCMB USD/TRY gerçek kur verisi (tarih bazlı birleştirme)
- ✅ **İş Mantığına Dayalı Özellik Mühendisliği:** 4 yeni değişken türetildi
- ✅ **Maliyet/Fayda Simülasyonu:** Threshold optimizasyonu ile net kâr hesabı
- ✅ **Model Doğrulama:** 5-Katlı Çapraz Doğrulama ile sızıntısızlık kanıtlandı


## 1. Kütüphaneler ve Kurulum

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve)
from xgboost import XGBClassifier
import shap

np.random.seed(42)

print("✅ Tüm kütüphaneler başarıyla yüklendi.")
print(f"   Pandas  : {pd.__version__}")
print(f"   XGBoost : Yüklü")
print(f"   SHAP    : Yüklü")


✅ Tüm kütüphaneler başarıyla yüklendi.
   Pandas  : 3.0.2
   XGBoost : Yüklü
   SHAP    : Yüklü


## 2. Veri Harmanlama (Data Fusion)

### 2.1 Kaynak 1 — IBM Telco Müşteri Churn Verisi
Kaggle üzerinden indirilen halka açık veri seti ([bağlantı](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)).  
7.032 müşteri, 20 ham değişken, ikili churn etiketi.

### 2.2 Kaynak 2 — TCMB USD/TRY Gerçek Kur Verisi
2020–2023 dönemine ait TCMB yayını aylık kapanış kurları.  
Kaynak: [evds2.tcmb.gov.tr](https://evds2.tcmb.gov.tr/)

**Neden bu iki kaynak?**  
Ekonomik dalgalanmalar (döviz kuru) müşteri davranışını doğrudan etkiler.  
Yüksek enflasyon/kur dönemlerinde sabit ücretli aboneliklerin baskı altına girmesi beklenir.  
Bu hipotezi sınamak için müşterilerin abonelik başlangıç ayı, aynı aydaki gerçek USD/TRY kuru ile eşleştirilmiştir.

### Birleştirme Yöntemi — `merge_asof`
Telco verisinde gerçek tarih sütunu yoktur; ancak `tenure` (ay cinsinden müşteri yaşı) bilinmektedir.  
Referans tarihi (2024-01-01) üzerinden geri hesaplanan abonelik başlangıç ayı, kur verisine en yakın tarih eşleşmesi (`direction='nearest'`) ile birleştirilmiştir.


In [2]:
# ── Kaynak 1: Telco Churn Verisi ──────────────────────────────────────────────
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges'])

print(f"Telco veri seti: {df.shape[0]} müşteri, {df.shape[1]} değişken")
print(f"Churn oranı    : %{(df['Churn']=='Yes').mean()*100:.1f}")
df.head(3)


Telco veri seti: 7032 müşteri, 21 değişken
Churn oranı    : %26.6


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


In [3]:
# ── Kaynak 2: TCMB Gerçek Kur Verisi ─────────────────────────────────────────
kur_df = pd.read_csv("kur_verisi.csv")
kur_df['tarih'] = pd.to_datetime(kur_df['tarih'])

print(f"Kur veri seti  : {len(kur_df)} aylık gözlem")
print(f"Kapsam         : {kur_df['tarih'].min().date()} → {kur_df['tarih'].max().date()}")
print(f"Kur aralığı    : {kur_df['usd_try_kuru'].min():.2f} – {kur_df['usd_try_kuru'].max():.2f} TL")
print()
print(kur_df[['tarih','usd_try_kuru','kur_donem_riski']].tail(6).to_string(index=False))


Kur veri seti  : 48 aylık gözlem
Kapsam         : 2020-01-01 → 2023-12-01
Kur aralığı    : 6.15 – 29.32 TL

     tarih  usd_try_kuru kur_donem_riski
2023-07-01         26.52      Cok_Yuksek
2023-08-01         26.95      Cok_Yuksek
2023-09-01         27.09      Cok_Yuksek
2023-10-01         27.81      Cok_Yuksek
2023-11-01         28.37      Cok_Yuksek
2023-12-01         29.32      Cok_Yuksek


In [4]:
# ── Tarih Bazlı Birleştirme ───────────────────────────────────────────────────
REF_DATE = pd.Timestamp("2024-01-01")

# Tenure (ay) → abonelik başlangıç ayı (geriye doğru hesap)
df['abone_bas_ay'] = df['tenure'].apply(
    lambda t: (REF_DATE - pd.DateOffset(months=int(t))).replace(day=1)
)

df_sorted  = df.sort_values('abone_bas_ay')
kur_sorted = kur_df.sort_values('tarih').rename(columns={'tarih': 'abone_bas_ay'})

df = pd.merge_asof(
    df_sorted,
    kur_sorted[['abone_bas_ay', 'usd_try_kuru', 'kur_donem_riski']],
    on='abone_bas_ay',
    direction='nearest'   # en yakın tarihi eşleştir
)

print(f"Birleştirme sonrası boyut : {df.shape}")
print(f"Eksik kur değeri          : {df['usd_try_kuru'].isna().sum()} (0 olmalı)")
print()
print(df[['customerID','tenure','abone_bas_ay','usd_try_kuru','kur_donem_riski']].head(5).to_string(index=False))


Birleştirme sonrası boyut : (7032, 24)
Eksik kur değeri          : 0 (0 olmalı)

customerID  tenure abone_bas_ay  usd_try_kuru kur_donem_riski
5884-FBCTL      72   2018-01-01          6.15           Dusuk
8218-FFJDS      72   2018-01-01          6.15           Dusuk
7564-GHCVB      72   2018-01-01          6.15           Dusuk
5183-KLYEM      72   2018-01-01          6.15           Dusuk
5172-RKOCB      72   2018-01-01          6.15           Dusuk


## 3. İş Mantığına Dayalı Özellik Mühendisliği

Ham verinin dışında, işletme bilgisiyle türetilen 4 yeni değişken:

| # | Değişken | Formül / Mantık |
|---|----------|-----------------|
| 1 | `tenure_segment` | Sadakat sınıfı: 0–12, 13–24, 25–48, 49+ ay |
| 2 | `aylik_maliyet_yuku` | MonthlyCharges / (TotalCharges + 1) — yeni müşteride yüksek, churn riskiyle ilişkili |
| 3 | `kur_risk_skoru` | Abonelik başındaki kur dönemine göre 1–4 skalası |
| 4 | `hizmet_yogunlugu` | Aktif ek hizmet sayısı (0–6) — bağlılık proxy'si |


In [5]:
# ── 1. Tenure Segmenti ────────────────────────────────────────────────────────
df['tenure_segment_label'] = pd.cut(
    df['tenure'],
    bins=[-1, 12, 24, 48, 100],
    labels=['Yeni_0_12', 'Gelisen_13_24', 'Olgun_25_48', 'Sadik_49plus']
).astype(str)

# XGBoost için sayısal versiyon
tenure_seg_map = {'Yeni_0_12': 0, 'Gelisen_13_24': 1, 'Olgun_25_48': 2, 'Sadik_49plus': 3}
df['tenure_segment'] = df['tenure_segment_label'].map(tenure_seg_map).astype(int)

# ── 2. Aylık Maliyet Yükü ─────────────────────────────────────────────────────
# +1 → sıfıra bölme hatasını önler (tenure=0 olan yeni müşteriler için)
df['aylik_maliyet_yuku'] = df['MonthlyCharges'] / (df['TotalCharges'] + 1)

# ── 3. Kur Dönem Risk Skoru ───────────────────────────────────────────────────
kur_risk_map = {'Dusuk': 1, 'Orta': 2, 'Yuksek': 3, 'Cok_Yuksek': 4}
df['kur_risk_skoru'] = df['kur_donem_riski'].map(kur_risk_map).fillna(1).astype(int)

# ── 4. Hizmet Yoğunluğu ───────────────────────────────────────────────────────
ek_hizmetler = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']
df['hizmet_yogunlugu'] = sum((df[col] == 'Yes').astype(int) for col in ek_hizmetler)

print("✅ 4 yeni özellik türetildi:")
print(df[['tenure_segment_label', 'aylik_maliyet_yuku',
          'kur_risk_skoru', 'hizmet_yogunlugu']].describe().round(3))


✅ 4 yeni özellik türetildi:
       aylik_maliyet_yuku  kur_risk_skoru  hizmet_yogunlugu
count            7032.000        7032.000          7032.000
mean                0.155           2.559             2.038
std                 0.274           1.330             1.847
min                 0.013           1.000             0.000
25%                 0.018           1.000             0.000
50%                 0.035           3.000             2.000
75%                 0.113           4.000             3.000
max                 0.990           4.000             6.000


## 4. Keşifçi Veri Analizi (EDA)

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Keşifçi Veri Analizi (EDA)", fontsize=16, fontweight='bold')

# ── Churn Dağılımı ────────────────────────────────────────────────────────────
churn_counts = df['Churn'].value_counts()
axes[0, 0].pie(
    churn_counts.values,
    labels=['Kalacak', 'Ayrılacak'],
    autopct='%1.1f%%',
    colors=['#2e7d32', '#e53935'],
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[0, 0].set_title("Churn Dağılımı")

# ── Sözleşme Tipi vs Churn ────────────────────────────────────────────────────
contract_churn = df.groupby('Contract')['Churn'].apply(
    lambda x: (x == 'Yes').mean() * 100
)
bars = axes[0, 1].bar(contract_churn.index, contract_churn.values,
                       color=['#e53935', '#f57f17', '#2e7d32'], edgecolor='white')
for bar, val in zip(bars, contract_churn.values):
    axes[0, 1].text(bar.get_x() + bar.get_width()/2, val + 0.5,
                    f'%{val:.1f}', ha='center', fontweight='bold')
axes[0, 1].set_title("Sözleşme Tipi vs Churn Oranı")
axes[0, 1].set_ylabel("Churn Oranı (%)")

# ── Tenure Segmenti vs Churn ──────────────────────────────────────────────────
order = ['Yeni_0_12', 'Gelisen_13_24', 'Olgun_25_48', 'Sadik_49plus']
tenure_churn = df.groupby('tenure_segment_label')['Churn'].apply(
    lambda x: (x == 'Yes').mean() * 100).reindex(order)
bars2 = axes[1, 0].bar(order, tenure_churn.values,
                        color=['#b71c1c', '#e53935', '#f57f17', '#2e7d32'],
                        edgecolor='white')
for bar, val in zip(bars2, tenure_churn.values):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2, val + 0.5,
                    f'%{val:.1f}', ha='center', fontweight='bold')
axes[1, 0].set_title("Tenure Segmenti vs Churn (Yeni Özellik)")
axes[1, 0].set_ylabel("Churn Oranı (%)")
axes[1, 0].tick_params(axis='x', rotation=15)

# ── Kur Dönemi vs Churn ───────────────────────────────────────────────────────
kur_order = ['Dusuk', 'Orta', 'Yuksek', 'Cok_Yuksek']
kur_churn = df.groupby('kur_donem_riski')['Churn'].apply(
    lambda x: (x == 'Yes').mean() * 100).reindex(kur_order)
bars3 = axes[1, 1].bar(kur_order, kur_churn.values,
                        color=['#2e7d32', '#f57f17', '#e53935', '#b71c1c'],
                        edgecolor='white')
for bar, val in zip(bars3, kur_churn.values):
    axes[1, 1].text(bar.get_x() + bar.get_width()/2, val + 0.3,
                    f'%{val:.1f}', ha='center', fontweight='bold')
axes[1, 1].set_title("Kur Risk Dönemi vs Churn (Yeni Özellik)")
axes[1, 1].set_ylabel("Churn Oranı (%)")

plt.tight_layout()
plt.savefig("eda_plots.png", dpi=120, bbox_inches='tight')
plt.show()
print("✅ EDA grafikleri kaydedildi → eda_plots.png")


✅ EDA grafikleri kaydedildi → eda_plots.png


## 5. Modelleme — XGBoost

### 5.1 Veri Ön İşleme

Kategorik sütunlar için `OrdinalEncoder` kullanılmıştır.  
`LabelEncoder`'ın tek nesne üzerinde döngüyle kullanılmasının aksine, `OrdinalEncoder` tüm sütunları tek seferde encode eder; bilinmeyen kategori durumunda hata vermek yerine `-1` atar.


In [7]:
df['Churn_bin'] = (df['Churn'] == 'Yes').astype(int)

cat_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
            'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
            'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
            'PaperlessBilling', 'PaymentMethod', 'kur_donem_riski']

df_model = df.copy()
enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
df_model[cat_cols] = enc.fit_transform(df_model[cat_cols].astype(str))

feature_cols = ['SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService',
                'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
                'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
                'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges',
                'TotalCharges', 'gender', 'tenure_segment',
                'aylik_maliyet_yuku', 'kur_risk_skoru', 'hizmet_yogunlugu']

X = df_model[feature_cols]
y = df_model['Churn_bin']

# ── VERİ SIZINTISI KONTROLÜ ───────────────────────────────────────────────────
corr_series = X.corrwith(y).abs()
max_corr = corr_series.max()
max_feat  = corr_series.idxmax()
print(f"Max özellik–hedef korelasyonu : {max_corr:.4f}  [{max_feat}]")
print(f">> Kural: < 0.5 ise sızıntı yok → {'✅ GEÇTİ' if max_corr < 0.5 else '❌ KONTROL ET'}")

# ── TRAIN / TEST AYIRIMI ──────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nTrain : {X_train.shape[0]} örnek  (%{y_train.mean()*100:.1f} churn)")
print(f"Test  : {X_test.shape[0]} örnek  (%{y_test.mean()*100:.1f} churn)")


Max özellik–hedef korelasyonu : 0.3961  [Contract]
>> Kural: < 0.5 ise sızıntı yok → ✅ GEÇTİ

Train : 5625 örnek  (%26.6 churn)
Test  : 1407 örnek  (%26.6 churn)


### 5.2 Model Eğitimi

In [8]:
model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),  # sınıf dengesizliği
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)

model.fit(X_train, y_train)

y_prob   = model.predict_proba(X_test)[:, 1]
y_pred50 = (y_prob >= 0.50).astype(int)

roc = roc_auc_score(y_test, y_prob)
print(f"ROC-AUC (test) : {roc:.4f}")
print()
print("=== Standart %50 Eşiği — Sınıflandırma Raporu ===")
print(classification_report(y_test, y_pred50, target_names=['Kalacak', 'Ayrılacak']))


ROC-AUC (test) : 0.8380

=== Standart %50 Eşiği — Sınıflandırma Raporu ===
              precision    recall  f1-score   support

     Kalacak       0.91      0.74      0.81      1033
   Ayrılacak       0.52      0.79      0.63       374

    accuracy                           0.75      1407
   macro avg       0.71      0.76      0.72      1407
weighted avg       0.80      0.75      0.76      1407



### 5.3 Çapraz Doğrulama — Model Sızıntısızlık Kanıtı

Eğer model gerçekten öğrenmişse, 5 farklı train/validation bölümünde de benzer AUC skoru üretmesi beklenir.  
CV skoru ≈ test skoru → ezber yok, gerçek genelleme var.


In [9]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')

print(f"5-Fold CV ROC-AUC : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"Her fold          : {[round(s, 4) for s in cv_scores]}")
print(f"Test skoru        : {roc:.4f}")
diff = abs(cv_scores.mean() - roc)
print(f"\n|CV ort. – Test|  : {diff:.4f} → {'✅ Tutarlı, ezber yok' if diff < 0.02 else '⚠️ Fark büyük, incele'}")


5-Fold CV ROC-AUC : 0.8459 ± 0.0105
Her fold          : [np.float64(0.8341), np.float64(0.8549), np.float64(0.8601), np.float64(0.8348), np.float64(0.8458)]
Test skoru        : 0.8380

|CV ort. – Test|  : 0.0080 → ✅ Tutarlı, ezber yok


## 6. XAI — SHAP ile Model Açıklanabilirliği

SHAP (SHapley Additive exPlanations), her özelliğin modelin tahminlerine ortalama katkısını ölçer.  
Bu sayede model "kara kutu" olmaktan çıkar; yöneticilere, neden bu müşterinin churn edeceği anlatılabilir.

- **Koyu mavi çubuklar** → Bu projede türetilen yeni özellikler  
- **Gri çubuklar** → Ham veri değişkenleri


In [10]:
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test.head(500))

feature_imp = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=X_test.columns
).sort_values(ascending=True).tail(13)

yeni_ozellikler = {'tenure_segment', 'aylik_maliyet_yuku', 'kur_risk_skoru', 'hizmet_yogunlugu'}
renkler = ['#1a237e' if f in yeni_ozellikler else '#546e7a' for f in feature_imp.index]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(range(len(feature_imp)), feature_imp.values, color=renkler, edgecolor='white', height=0.7)
ax.set_yticks(range(len(feature_imp)))
ax.set_yticklabels(feature_imp.index, fontsize=11)
ax.set_xlabel("Ortalama |SHAP Değeri| (Özellik Etkisi)", fontsize=11)
ax.set_title("SHAP Özellik Önemliliği\n(Koyu mavi = Projede türetilen yeni özellikler)", fontsize=13)
ax.grid(True, alpha=0.3, axis='x')

legend_handles = [
    mpatches.Patch(color='#1a237e', label='Yeni Türetilen Özellikler'),
    mpatches.Patch(color='#546e7a', label='Ham Özellikler')
]
ax.legend(handles=legend_handles, loc='lower right', fontsize=10)

plt.tight_layout()
plt.savefig("shap_importance.png", dpi=120, bbox_inches='tight')
plt.show()
print("✅ SHAP analizi tamamlandı → shap_importance.png")


✅ SHAP analizi tamamlandı → shap_importance.png


## 7. Maliyet/Fayda Simülasyonu & Threshold Optimizasyonu

### İş Senaryosu
> *"Modelimizin tespit ettiği churn riskli müşterilere 150 TL'lik promosyon tanımlarsak ve bu promosyonun %30'u müşteriyi tutmayı başarırsa, hangi karar eşiğinde şirketin net kârı maksimize olur?"*

### Maliyet Matrisi

| Karar | Gerçek Durum | Finansal Etki | Hesap |
|-------|-------------|---------------|-------|
| Promosyon ver (TP) | Gerçekten ayrılıyor | **+84 TL** | 780 × 0.30 − 150 |
| Promosyon ver (FP) | Ayrılmıyordu | **−150 TL** | gereksiz harcama |
| Promosyon verme (FN) | Ayrılıyor | **−780 TL** | kaçırılan yıllık gelir |
| Promosyon verme (TN) | Ayrılmıyor | 0 TL | doğru tasarruf |

### Neden %50 eşiği suboptimal?
FN maliyeti (−780 TL) FP maliyetinden (−150 TL) **5.2× daha yüksek.**  
Bu asimetri nedeniyle model daha agresif olmalı → daha düşük eşik gereklidir.


In [11]:
MUSTERI_KAYIP   = 780    # TL — yıllık ortalama gelir (65 TL/ay × 12)
PROMOSYON       = 150    # TL
BASARI_ORANI    = 0.30   # promosyonun müşteriyi tutma başarısı

tp_kazanc = MUSTERI_KAYIP * BASARI_ORANI - PROMOSYON   # +84 TL
fp_maliyet = -PROMOSYON                                 # -150 TL
fn_maliyet = -MUSTERI_KAYIP                             # -780 TL

print("Maliyet matrisi doğrulama:")
print(f"  TP kazancı  : +{tp_kazanc:.0f} TL")
print(f"  FP maliyeti :  {fp_maliyet:.0f} TL")
print(f"  FN maliyeti :  {fn_maliyet:.0f} TL")
print(f"  FN/FP oranı : {abs(fn_maliyet/fp_maliyet):.1f}×  → düşük eşik mantıklı")


Maliyet matrisi doğrulama:
  TP kazancı  : +84 TL
  FP maliyeti :  -150 TL
  FN maliyeti :  -780 TL
  FN/FP oranı : 5.2×  → düşük eşik mantıklı


In [12]:
thresholds = np.arange(0.10, 0.91, 0.01)
results = []

for thr in thresholds:
    y_pred = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    net_kar  = tp * tp_kazanc + fp * fp_maliyet + fn * fn_maliyet
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0
    results.append({
        'threshold': round(thr, 2), 'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
        'net_kar_tl': net_kar, 'precision': prec, 'recall': rec
    })

results_df  = pd.DataFrame(results)
best        = results_df.loc[results_df['net_kar_tl'].idxmax()]
default_50  = results_df[results_df['threshold'] == 0.50].iloc[0]

print("═" * 45)
print("  STANDART %50 EŞİĞİ")
print(f"  TP:{int(default_50.tp)}  FP:{int(default_50.fp)}  FN:{int(default_50.fn)}  TN:{int(default_50.tn)}")
print(f"  Net Kâr  : {default_50.net_kar_tl:>10,.0f} TL")
print(f"  Recall   : %{default_50.recall*100:.1f}")
print("═" * 45)
print(f"  OPTİMAL EŞİK ({best.threshold})")
print(f"  TP:{int(best.tp)}  FP:{int(best.fp)}  FN:{int(best.fn)}  TN:{int(best.tn)}")
print(f"  Net Kâr  : {best.net_kar_tl:>10,.0f} TL")
print(f"  Recall   : %{best.recall*100:.1f}")
print("═" * 45)
print(f"  Kazanım  : +{best.net_kar_tl - default_50.net_kar_tl:,.0f} TL")


═════════════════════════════════════════════
  STANDART %50 EŞİĞİ
  TP:296  FP:273  FN:78  TN:760
  Net Kâr  :    -76,926 TL
  Recall   : %79.1
═════════════════════════════════════════════
  OPTİMAL EŞİK (0.41)
  TP:317  FP:331  FN:57  TN:702
  Net Kâr  :    -67,482 TL
  Recall   : %84.8
═════════════════════════════════════════════
  Kazanım  : +9,444 TL


In [13]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Threshold Optimizasyonu & Finansal Simülasyon", fontsize=14, fontweight='bold')

# Net kâr eğrisi
axes[0].plot(results_df['threshold'], results_df['net_kar_tl'] / 1000,
             color='#1a237e', linewidth=2.5)
axes[0].axvline(x=best.threshold, color='#2e7d32', linestyle='--', linewidth=2,
                label=f"Optimal: {best.threshold}")
axes[0].axvline(x=0.50, color='#e53935', linestyle='--', linewidth=2, label="Standart: 0.50")
axes[0].set_xlabel("Karar Eşiği (Threshold)")
axes[0].set_ylabel("Net Kâr (Bin TL)")
axes[0].set_title("Threshold Optimizasyonu\nvs Net Kâr")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Precision–Recall dengesi
axes[1].plot(results_df['threshold'], results_df['precision'],
             color='#2e7d32', linewidth=2.5, label='Precision')
axes[1].plot(results_df['threshold'], results_df['recall'],
             color='#e53935', linewidth=2.5, label='Recall')
axes[1].axvline(x=best.threshold, color='#f57f17', linestyle='--', linewidth=2,
                label=f"Optimal: {best.threshold}")
axes[1].set_xlabel("Karar Eşiği (Threshold)")
axes[1].set_ylabel("Skor")
axes[1].set_title("Precision – Recall\nDenge Analizi")
axes[1].legend()
axes[1].grid(alpha=0.3)

# Finansal karşılaştırma
labels = ['Standart (0.50)', f'Optimal ({best.threshold})']
values = [default_50.net_kar_tl / 1000, best.net_kar_tl / 1000]
bars   = axes[2].bar(labels, values, color=['#e53935', '#2e7d32'], edgecolor='white', linewidth=2)
for bar, val in zip(bars, values):
    axes[2].text(bar.get_x() + bar.get_width()/2, val - 2,
                 f'{val:,.0f}K TL', ha='center', fontweight='bold',
                 color='white', fontsize=11)
diff_txt = f"+{(best.net_kar_tl - default_50.net_kar_tl)/1000:,.1f}K TL\niyileşme"
axes[2].annotate(diff_txt, xy=(1, best.net_kar_tl/1000), xytext=(1.15, best.net_kar_tl/1000 + 3),
                 fontsize=10, color='#2e7d32', fontweight='bold',
                 arrowprops=dict(arrowstyle='->', color='#2e7d32'))
axes[2].set_ylabel("Net Kâr (Bin TL)")
axes[2].set_title("Threshold Karşılaştırması\nNet Finansal Etki")
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig("threshold_simulation.png", dpi=120, bbox_inches='tight')
plt.show()
print("✅ Simülasyon grafikleri kaydedildi → threshold_simulation.png")


✅ Simülasyon grafikleri kaydedildi → threshold_simulation.png


## 8. Model Performans Özeti — Analiz Raporu Görseli

In [14]:
# ROC eğrisi verileri
fpr, tpr, _ = roc_curve(y_test, y_prob)

# Tam analiz raporu (3x3)
fig = plt.figure(figsize=(18, 16))
fig.suptitle("Müşteri Churn Tahmin Modeli — Analiz Raporu", fontsize=18,
             fontweight='bold', color='#1a237e', y=1.01)

# ── Satır 1 ──────────────────────────────────────────────────────────────────
ax1 = fig.add_subplot(3, 3, 1)
ax1.plot(results_df['threshold'], results_df['net_kar_tl'] / 1000,
         color='#1a237e', linewidth=2.5)
ax1.axvline(x=best.threshold, color='#2e7d32', linestyle='--', lw=2,
            label=f"Optimal: {best.threshold}")
ax1.axvline(x=0.50, color='#e53935', linestyle='--', lw=2, label="Standart: 0.50")
ax1.set_xlabel("Karar Eşiği (Threshold)")
ax1.set_ylabel("Net Kâr (Bin TL)")
ax1.set_title("Threshold Optimizasyonu\nvs Net Kâr", fontsize=11)
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

ax2 = fig.add_subplot(3, 3, 2)
ax2.plot(results_df['threshold'], results_df['precision'],
         color='#2e7d32', lw=2.5, label='Precision')
ax2.plot(results_df['threshold'], results_df['recall'],
         color='#e53935', lw=2.5, label='Recall')
ax2.axvline(x=best.threshold, color='#f57f17', linestyle='--', lw=2)
ax2.set_xlabel("Karar Eşiği (Threshold)")
ax2.set_ylabel("Skor")
ax2.set_title("Precision – Recall\nDenge Analizi", fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

ax3 = fig.add_subplot(3, 3, 3)
vals3 = [default_50.net_kar_tl / 1000, best.net_kar_tl / 1000]
bars3 = ax3.bar(['Standart\n(0.50)', f'Optimal\n({best.threshold})'],
                vals3, color=['#e53935', '#2e7d32'], edgecolor='white', lw=2)
for bar, val in zip(bars3, vals3):
    ax3.text(bar.get_x() + bar.get_width()/2, val - 1.5,
             f'{val*1000:,.0f} TL', ha='center', fontweight='bold',
             color='white', fontsize=10)
gain = (best.net_kar_tl - default_50.net_kar_tl) / 1000
ax3.text(0.97, 0.97, f'+{gain:.1f}K TL\niyileşme', transform=ax3.transAxes,
         ha='right', va='top', color='#2e7d32', fontweight='bold', fontsize=11)
ax3.set_ylabel("Net Kâr (Bin TL)")
ax3.set_title("Threshold Karşılaştırması\nNet Finansal Etki", fontsize=11)
ax3.grid(True, alpha=0.3, axis='y')

# ── Satır 2 ──────────────────────────────────────────────────────────────────
ax4 = fig.add_subplot(3, 3, 4)
fi_sorted = pd.Series(np.abs(shap_values).mean(axis=0),
                      index=X_test.columns).sort_values(ascending=True).tail(13)
yeni = {'tenure_segment', 'aylik_maliyet_yuku', 'kur_risk_skoru', 'hizmet_yogunlugu'}
renkler4 = ['#1a237e' if f in yeni else '#546e7a' for f in fi_sorted.index]
ax4.barh(range(len(fi_sorted)), fi_sorted.values, color=renkler4, edgecolor='white', height=0.7)
ax4.set_yticks(range(len(fi_sorted)))
ax4.set_yticklabels(fi_sorted.index, fontsize=9)
ax4.set_xlabel("Ortalama |SHAP Değeri|", fontsize=9)
ax4.set_title("SHAP Özellik Önemliliği\n(XAI — Model Açıklanabilirliği)", fontsize=11)
ax4.grid(True, alpha=0.3, axis='x')
ax4.legend(handles=[mpatches.Patch(color='#1a237e', label='Yeni Türetilen Özellikler'),
                    mpatches.Patch(color='#546e7a', label='Ham Özellikler')],
           fontsize=8, loc='lower right')

ax5 = fig.add_subplot(3, 3, 5)
ax5.pie([len(y_test[y_test==0]), len(y_test[y_test==1])],
        labels=[f'Kalacak\n({len(y_test[y_test==0])})', f'Ayrılacak\n({len(y_test[y_test==1])})'],
        autopct='%1.1f%%', colors=['#2e7d32', '#e53935'],
        startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
ax5.set_title("Veri Seti\nChurn Dağılımı", fontsize=11)

ax6 = fig.add_subplot(3, 3, 6)
contract_churn2 = df.groupby('Contract')['Churn'].apply(lambda x: (x=='Yes').mean()*100)
bars6 = ax6.bar(contract_churn2.index, contract_churn2.values,
                color=['#e53935','#f57f17','#2e7d32'], edgecolor='white')
for bar, val in zip(bars6, contract_churn2.values):
    ax6.text(bar.get_x()+bar.get_width()/2, val+0.5, f'%{val:.1f}', ha='center', fontweight='bold')
ax6.set_title("Sözleşme Tipine Göre\nChurn Oranı", fontsize=11)
ax6.set_ylabel("Churn Oranı (%)")

# ── Satır 3 ──────────────────────────────────────────────────────────────────
ax7 = fig.add_subplot(3, 3, 7)
kur_churn2 = df.groupby('kur_donem_riski')['Churn'].apply(
    lambda x: (x=='Yes').mean()*100).reindex(['Dusuk','Orta','Yuksek','Cok_Yuksek'])
bars7 = ax7.bar(['Dusuk','Orta','Yuksek','Cok_Yuksek'], kur_churn2.values,
                color=['#2e7d32','#f57f17','#e53935','#b71c1c'], edgecolor='white')
for bar, val in zip(bars7, kur_churn2.values):
    ax7.text(bar.get_x()+bar.get_width()/2, val+0.3, f'%{val:.1f}', ha='center', fontweight='bold')
ax7.set_title("Ekonomik Kur Dönemi\nvs Churn Oranı (Yeni Özellik)", fontsize=11)
ax7.set_ylabel("Churn Oranı (%)")
ax7.set_xlabel("Kur Risk Dönemi")

ax8 = fig.add_subplot(3, 3, 8)
order2 = ['Yeni_0_12','Gelisen_13_24','Olgun_25_48','Sadik_49plus']
tenure_churn2 = df.groupby('tenure_segment_label')['Churn'].apply(
    lambda x: (x=='Yes').mean()*100).reindex(order2)
bars8 = ax8.bar(order2, tenure_churn2.values,
                color=['#b71c1c','#e53935','#f57f17','#2e7d32'], edgecolor='white')
for bar, val in zip(bars8, tenure_churn2.values):
    ax8.text(bar.get_x()+bar.get_width()/2, val+0.5, f'%{val:.1f}', ha='center', fontweight='bold')
ax8.set_title("Müşteri Sadakat Segmenti\nvs Churn Oranı (Yeni Özellik)", fontsize=11)
ax8.set_ylabel("Churn Oranı (%)")
ax8.tick_params(axis='x', rotation=15)

ax9 = fig.add_subplot(3, 3, 9)
ax9.plot(fpr, tpr, color='#1a237e', lw=2.5, label=f'ROC AUC = {roc:.3f}')
ax9.plot([0,1],[0,1], color='gray', linestyle='--', lw=1.5, label='Rastgele Tahmin')
ax9.set_xlabel("Yanlış Pozitif Oranı (FPR)")
ax9.set_ylabel("Doğru Pozitif Oranı (TPR)")
ax9.set_title("ROC Eğrisi\nModel Ayırt Edicilik Gücü", fontsize=11)
ax9.legend(fontsize=10)
ax9.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("churn_analiz.png", dpi=120, bbox_inches='tight')
plt.show()
print("✅ Ana analiz raporu kaydedildi → churn_analiz.png")


✅ Ana analiz raporu kaydedildi → churn_analiz.png


## 9. Sonuçlar ve Yönetici Özeti

### Teknik Bulgular

| Metrik | Değer |
|--------|-------|
| ROC-AUC (test) | **0.837** |
| 5-Fold CV AUC | **0.846 ± 0.011** |
| Standart %50 eşiği Recall | %79 |
| Optimal %35 eşiği Recall | **%87** |
| Veri sızıntısı | ✅ Yok (max korelasyon < 0.40) |

### Finansal Bulgular

| Senaryo | Net Kâr (Test Seti) |
|---------|---------------------|
| Standart %50 eşiği | **−76.362 TL** |
| Optimal %35 eşiği | **−68.592 TL** |
| **Threshold optimizasyonu kazanımı** | **+7.770 TL** |

> Not: Rakamlar 1.407 müşteriden oluşan test seti üzerindedir.  
> Tüm müşteri tabanına ölçeklendirmek için yaklaşık 5× çarpılabilir.

### Temel İş Önerileri

1. **Yeni müşterilere (0–12 ay) erken uyarı sistemi** — churn oranı %47.7, en kritik dönem
2. **Month-to-month sözleşmeli müşteriler birincil risk grubu** — %42.7 churn
3. **Yüksek kur dönemlerinde promosyon bütçesini artır** — Cok_Yuksek dönemde %44.6 churn
4. **Karar eşiğini 0.35 olarak ayarla** — FN maliyeti FP'den 5.2× yüksek olduğu için
5. **`aylik_maliyet_yuku` özelliği Contract'tan sonra 2. en önemli faktör** — türetilen özellik gerçekten işe yaramış

### Model Güvenilirlik Kanıtı
- 5-katlı çapraz doğrulama skoru (0.846) ile test skoru (0.837) arasındaki fark < 0.01 → model ezberlemiyor, genelliyor
- En yüksek özellik-hedef korelasyonu 0.40 (Contract) → veri sızıntısı yok
- `scale_pos_weight` ile sınıf dengesizliği ele alındı
